# GCN-3 Ensemble Mirror Sensitivity Addendum

This inference-only notebook tests whether the selected five-member GCN-3 ensemble changes its stiffness prediction when a physically equivalent lattice is reflected. It reuses the saved final ensemble and does not retrain models or generate new lattices.

For every held-out test and external prediction lattice it evaluates: original coordinates, horizontal reflection (`mirror_x`), vertical reflection (`mirror_y`), and reflection of both coordinates (`mirror_xy`, equivalent to a 180-degree rotation). Reflections are taken about each lattice's bounding-box midpoint. Connectivity and edge weights are unchanged, while all coordinate-derived features are recomputed.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'
if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone skipped outside Colab.')

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
ENSEMBLE_RUN_DIR = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_uncertainty/run_final_ensemble_uncertainty_v1'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_mirror_sensitivity'
RESUMABLE_RUN_NAME = 'run_mirror_sensitivity_v1'
PUSH_RESULTS_TO_GITHUB = False
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_BRANCH = 'main'
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/gcn3_ensemble_mirror_sensitivity'

repo_root = Path(REPO_DIR).resolve() if IN_COLAB else Path.cwd().resolve()
pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
gnn_root = pipeline_root / 'gnn_prototype'
optimization_dir = gnn_root / 'GCN_Optimization'
required_files = (
    'colab_gnn_stiffness_prototype.py',
    'architecture_comparison_runner.py',
    'mirror_sensitivity_runner.py',
)
missing = [name for name in required_files if not (optimization_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing mirror-analysis modules under {optimization_dir}: {missing}')
os.chdir(pipeline_root)
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE or str(ENSEMBLE_RUN_DIR).startswith('/content/drive')):
    from google.colab import drive
    drive.mount('/content/drive')
if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'
if IN_COLAB:
    ensemble_run_dir = Path(ENSEMBLE_RUN_DIR)
else:
    latest_path = gnn_root / 'outputs' / 'gcn3_ensemble_uncertainty' / 'latest_run.txt'
    ensemble_run_dir = Path(latest_path.read_text(encoding='utf-8').strip())
    if not ensemble_run_dir.is_absolute():
        ensemble_run_dir = repo_root / ensemble_run_dir
if not (ensemble_run_dir / 'gcn3_ensemble_run_metadata.json').is_file():
    raise FileNotFoundError(f'Final ensemble artifacts not found at {ensemble_run_dir}')
if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/gcn3_ensemble_mirror_sensitivity') if IN_COLAB else gnn_root / 'outputs' / 'gcn3_ensemble_mirror_sensitivity'
output_root.mkdir(parents=True, exist_ok=True)
git_output_root = repo_root / GIT_RESULTS_SUBDIR
if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email: ').strip()
print(f'Final ensemble: {ensemble_run_dir}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')

In [ ]:
import pandas as pd
import shutil
import subprocess
import torch
from IPython.display import Image, display
from mirror_sensitivity_runner import MirrorSensitivityConfig, run_mirror_sensitivity_experiment

In [ ]:
MEMBER_SEEDS = (11, 42, 73, 101, 202)
SPLIT_SEED = 42
TRANSFORMS = ('original', 'mirror_x', 'mirror_y', 'mirror_xy')
BATCH_SIZE = 16
RESUME = True
config = MirrorSensitivityConfig(
    member_seeds=MEMBER_SEEDS,
    split_seed=SPLIT_SEED,
    transforms=TRANSFORMS,
    batch_size=BATCH_SIZE,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    resume=RESUME,
)
run_dir = output_root / RESUMABLE_RUN_NAME
print(config)
print(f'Output directory: {run_dir}')

In [ ]:
result = run_mirror_sensitivity_experiment(
    config,
    ensemble_run_dir=ensemble_run_dir,
    train_root=train_root,
    predict_root=predict_root,
    run_dir=run_dir,
)
output_dir = result['output_dir']
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')
print('Mirror sensitivity summary:')
display(result['summary'])
print('External prediction changes:')
display(pd.read_csv(output_dir / 'mirror_sensitivity_prediction_samples.csv'))

In [ ]:
for figure_name in ('mirror_original_vs_transformed.png', 'mirror_prediction_changes.png'):
    print(figure_name)
    display(Image(filename=str(output_dir / figure_name)))
print((output_dir / 'mirror_sensitivity_report.txt').read_text(encoding='utf-8'))

In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for Colab.')
    git_output_root.mkdir(parents=True, exist_ok=True)
    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)
    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'], check=True, capture_output=True, text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)
    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'], check=False)
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            message = f'Add GCN-3 ensemble mirror-sensitivity results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)
else:
    print('GitHub push disabled.')

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = shutil.make_archive('/content/gcn3_ensemble_mirror_sensitivity', 'zip', root_dir=output_dir)
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')